# Traces

The spans of traces as a waterfall: a lane per span under the span that called it, its
duration beside it, and the **critical path** (what the end of the request waited for) as a
dark line. Lanes fold: click a span's name; double-click to zoom to it; `[` and `]` fold and
unfold all. Docs: <https://mluttikh.github.io/anytimeline/how-to/traces.html>

In [ ]:
import numpy as np

from anytimeline import Timeline
from anytimeline.synthetic import traces

tr = traces(12, seed=3)  # .spans and .events: dicts of columns, as read_otlp gives them

## Many traces, folded

`expand=1` shows the roots only: one lane per request. The events in the spans (exceptions,
retries, cache misses) go on their spans' lanes.

In [ ]:
plot = Timeline(title="Twelve requests", height=420)
plot.spans(tr.spans, color="service", tooltip=["service", "status"], expand=1)
plot.events(tr.events, time="time", lane="span_id", symbol="name", color="#333333",
            tooltip=["attributes"], name="span events")
plot

## One trace

A request with a database timeout and a retry: the critical path runs through both. Switch
to **Order** in the toolbar to see the order of the calls however long each took.

In [ ]:
bad = tr.spans["trace_id"][list(tr.spans["status"]).index("error")]
spans = {k: v[tr.spans["trace_id"] == bad] for k, v in tr.spans.items()}
events = {k: v[tr.events["trace_id"] == bad] for k, v in tr.events.items()}

one = Timeline(title="A request with a retry")
one.spans(spans, color="service", tooltip=["service", "status"])
one.events(events, time="time", lane="span_id", symbol="name", color="#333333",
           tooltip=["attributes"], name="span events")
one

## Traces by endpoint

`group=` puts the traces of each endpoint under a header of their own.

In [ ]:
roots = tr.spans["parent_id"] == None  # noqa: E711
endpoint_of = dict(zip(tr.spans["trace_id"][roots], tr.spans["name"][roots], strict=True))
endpoint = np.array([endpoint_of[t] for t in tr.spans["trace_id"]], dtype=object)

by_endpoint = Timeline(title="Traces by endpoint", height=420)
by_endpoint.spans({**tr.spans, "endpoint": endpoint}, color="service", group="endpoint",
                  expand=1)
by_endpoint

## From OpenTelemetry

`read_otlp` reads OTLP/JSON: a file the OpenTelemetry Collector's file exporter wrote (one
export request per line), a file Jaeger or Tempo exported, or the parsed JSON. A small
export request here, written by hand:

In [ ]:
from anytimeline.otel import read_otlp

request = {"resourceSpans": [{
    "resource": {"attributes": [{"key": "service.name", "value": {"stringValue": "checkout"}}]},
    "scopeSpans": [{"spans": [
        {"traceId": "5b8efff798038103d269b633813fc60c", "spanId": "eee19b7ec3c1b174",
         "name": "POST /checkout", "kind": 2, "status": {"code": 2, "message": "declined"},
         "startTimeUnixNano": "1760256000000000000", "endTimeUnixNano": "1760256000250000000",
         "events": [{"timeUnixNano": "1760256000200000000", "name": "exception"}]},
        {"traceId": "5b8efff798038103d269b633813fc60c", "spanId": "0000000000000001",
         "parentSpanId": "eee19b7ec3c1b174", "name": "charge card", "kind": 3,
         "startTimeUnixNano": "1760256000010000000", "endTimeUnixNano": "1760256000200000000"},
    ]}],
}]}
otlp = read_otlp(request)  # or read_otlp("traces.jsonl")
Timeline(title="From OTLP").spans(otlp.spans, color="status").events(
    otlp.events, time="time", lane="span_id", symbol="name")